In [ ]:
# pandas
import pandas as pd

# json
import re
import json
from tqdm import tqdm

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

# DATASET

In [ ]:
# LOAD DATASET (REELS)

data = pd.read_csv("/kaggle/input/datasets/filippotenani/testing-llm/uniting_reels_engineered (1).csv")

In [ ]:
# LOAD DATASET (STORIES)

data = pd.read_csv("/kaggle/input/datasets/filippotenani/testing-llm/uniting_stories_engineered (1).csv")

In [ ]:
data.columns

In [ ]:
data.shape

In [ ]:
data["Post caption"].isna().sum()

In [ ]:
data["Filename"].value_counts()

# GEMINI

In [ ]:
!pip install -q google-genai pydantic
from kaggle_secrets import UserSecretsClient
from google import genai
from google.genai import types
from pydantic import BaseModel
from typing import Literal
import time

In [ ]:
# NOTE: PROMPT AND PYDANTIC MUST BE EXECUTEDFIRST BEFORE RUNNING THE FOLLOWING CODE!

## CAPTION

In [ ]:
# LLM SETUP

# gemini api
gemini_api_key = UserSecretsClient().get_secret("APIGEMINIFLASH")
client = genai.Client(api_key=gemini_api_key)

In [ ]:
# LLM CALL

def call_model(caption):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        # receives the caption
        contents=caption,
        config=types.GenerateContentConfig(
            # receives the system prompt
            system_instruction=system_prompt,
            response_mime_type="application/json",
            response_schema=PostFeatures,
            # no creativity, we want consistent responses
            temperature=0.1,
            # we saw that model already uses a lot of output tokens for thinking
            # so we need to increase the max
            max_output_tokens=1000
        )
    )
    return response.parsed.model_dump()

In [ ]:
# FEATURE EXTRACTION

# define the function that extracts the features
def extract_features(caption):
    # calls call_model and returns a dict
    try:
        # call_model now returns a dict directly, no need to parse JSON
        return call_model(caption)
    # if the model call fails entirely, return ERROR for both fields
    except Exception as e:
        print(f"Error: {e}")
        return {field: "ERROR" for field in PostFeatures.model_fields}

In [ ]:
# EXECUTE

# take Filename (unique id of every row) and Post caption and put them in a separate dataset
# then call the extract_feature function which itself calls call_model and parse_response
# save the results in a pandas dataframe

results = data[["Filename", "Post caption"]].copy()
for field in PostFeatures.model_fields:
    results[field] = "ERROR"

for idx, row in tqdm(results.iterrows(), total=len(results)):
    caption = row["Post caption"]
    if pd.isna(caption):
        continue
    extracted = extract_features(caption)
    for field, value in extracted.items():
        results.at[idx, field] = value
    print(f"{idx}: {extracted}")
    # 5 requests per minute = 1 every 12s, 15s to be safe
    time.sleep(15)

results.to_csv("extracted_features.csv", index=False)

In [ ]:
# EXECUTE: TEST ON 5 ROWS

# let's test the code on just 5 rows
# this block is just like the EXECUTE block but it is only applied to 5 rows

test_data = data[["Filename", "Post caption"]].head(5).copy()
for field in PostFeatures.model_fields:
    test_data[field] = "ERROR"

for idx, row in test_data.iterrows():
    caption = row["Post caption"]
    if pd.isna(caption):
        continue
    extracted = extract_features(caption)
    for field, value in extracted.items():
        test_data.at[idx, field] = value
    print(f"{idx}: {extracted}")
    # 5 requests per minute = 1 every 12s, 15s to be safe
    time.sleep(15)

print(test_data[["Filename"] + list(PostFeatures.model_fields)])

## VIDEO

In [ ]:
from tenacity import retry, wait_exponential, stop_after_attempt
import time

In [ ]:
# VERTEX AI SETUP

# setup vertex ai, the google cloud platform for running ai models
client_vertex = genai.Client(
    vertexai=True,
    project="project-997a2ade-117b-4af1-926",
    location="us-central1",
)

# gcs bucket and folder where the videos are stored
GCS_BUCKET = "afb-uniting-videos"
GCS_PREFIX = "videos"

In [ ]:
# LLM CALL VIDEO

# if the function fails, tenacity will automatically retry it instead of crashing
# wait_exponential: wait 30s before first retry, then double each time (30s, 60s, 120s)
# stop_after_attempt: give up after 5 failed attempts
# reraise: after 5 attempts, raise the original error instead of a tenacity error
@retry(wait=wait_exponential(multiplier=10, min=30, max=120), stop=stop_after_attempt(5), reraise=True)
def call_model_video(filename):
    # build the address pointing to the video on google cloud storage
    # gemini fetches it directly from there
    gcs_uri = f"gs://{GCS_BUCKET}/{GCS_PREFIX}/{filename}.mp4"
    file_part = types.Part.from_uri(file_uri=gcs_uri, mime_type="video/mp4")

    # llm call: send the video location and the prompt to the model
    response = client_vertex.models.generate_content(
        model="gemini-2.5-flash",
        # only the video goes in contents
        contents=[file_part],
        config=types.GenerateContentConfig(
            # system prompt goes here
            system_instruction=system_prompt_video,
            response_mime_type="application/json",
            # pydantic schema
            response_schema=PostFeatures,
            # no creativity, we want consistent responses
            temperature=0.1,
            # we saw that model already uses a lot of output tokens for thinking
            # so we need to increase the max
            max_output_tokens=1000
        )
    )

    return response.parsed.model_dump()

In [ ]:
# FEATURE EXTRACTION VIDEO

# calls call_model_video and parses the JSON response
def extract_features_video(filename):
    try:
        # call_model now returns a dict directly, no need to parse JSON
        return call_model_video(filename)
    # if the model call fails entirely (even after all tenacity retries), return ERROR for both fields
    except Exception as e:
        print(f"Error on {filename}: {e}")
        return {field: "ERROR" for field in PostFeatures.model_fields}

In [ ]:
# EXECUTE

# take Filename (unique id of every row) and put it in a separate dataset
# then call extract_features_video which uploads the video, calls the model and parses the response
# save the results in a pandas dataframe

results_video = data[["Filename"]].copy()
for field in PostFeatures.model_fields:
    results_video[field] = "ERROR"

for idx, row in tqdm(results_video.iterrows(), total=len(results_video)):
    extracted = extract_features_video(row["Filename"])
    for field, value in extracted.items():
        results_video.at[idx, field] = value
    print(f"{idx}: {extracted}")
    # sleep between calls to respect the API rate limit
    time.sleep(30)

results_video.to_csv("extracted_features_video.csv", index=False)

In [ ]:
# WE TEST THE VIDEOS ON 5 STORIES BECAUSE THEY HAVE AUDIO TO TRANSCRIBE
# IN THE REELS I SAW THE AUDIO WAS ONLY MUSIC

In [ ]:
# EXECUTE: TEST ON 5 VIDEOS

test_filenames = ["101_457_3215", "101_457_3217", "101_457_3219", "101_457_3220", "101_457_3221"]
test_data = pd.DataFrame({"Filename": test_filenames})
for field in PostFeatures.model_fields:
    test_data[field] = "ERROR"

for idx, row in test_data.iterrows():
    extracted = extract_features_video(row["Filename"])
    for field, value in extracted.items():
        test_data.at[idx, field] = value
    print(f"{idx}: {extracted}")
    # sleep between calls to respect the API rate limit
    time.sleep(30)

print(test_data)

# PROMPT LIBRARY

## CAPTION (LENGTH)

In [ ]:
# LENGTH
# Word count of the caption

data["caption_length"] = data["Post caption"].str.split().str.len()
data[["Post caption", "caption_length"]].head(10)

## CAPTION (TONE CAPTION, FUNNEL CAPTION)

In [ ]:
system_prompt = """
=== RUOLO ===
Sei un esperto analista di marketing nel campo dei social media, con vasta esperienza nell'analisi
di tutti i tipi di dati social. Sei un professionista molto orientato ai dati, capace di identificare
informazioni rilevanti da post, descrizioni e diversi tipi di dati di marketing e social media.

=== CONTESTO ===
Aiuterai un gruppo di studenti in un progetto di data science focalizzato sull'analisi di dati social
media e su cosa influenza l'engagement. La maggior parte delle feature di questo dataset sono non
strutturate, il che significa che non possono essere facilmente utilizzate in un modello statistico
tradizionale. Diventa quindi fondamentale estrarre feature strutturate e significative da quelle non strutturate.

=== TASK ===
Ti verrà fornita la caption di un post Instagram o TikTok. Il tuo obiettivo è estrarre esattamente
le seguenti variabili dalla caption, seguendo attentamente le descrizioni e le categorie fornite per ciascuna.

=== VARIABILI DA ESTRARRE ===

tone_caption: la valenza emotiva e il registro affettivo percepito nel testo della caption.
Non è il tono strategico del brand, ma come il testo ti fa sentire leggendolo. Scegli UNA categoria:

- "entusiasta/energico": il testo trasmette energia positiva e vivacità. Sono presenti
  esclamazioni frequenti, maiuscole enfatiche, ritmo incalzante, aggettivi forti. Il registro
  è ad alta intensità emotiva positiva, trasmette eccitazione e coinvolgimento.
  Esempi: "Ragazzi è PAZZESCO, non ci posso credere!!!", uso abbondante di punti esclamativi,
  aggettivi come "incredibile", "assurdo", "pazzesco", frasi brevi e energiche che si
  susseguono rapidamente.

- "ironico/scherzoso": il testo usa umorismo, autoironia, situazioni assurde o paradossali,
  battute, doppi sensi come registro comunicativo principale. Il tono è leggero e non si
  prende sul serio, usando la comicità come leva per coinvolgere il lettore e rendere
  il messaggio promozionale più naturale e accessibile.
  Esempi: caption con doppi sensi, battute sulla vita quotidiana, tono canzonatorio,
  situazioni volutamente assurde raccontate con leggerezza, humor sul rapporto di coppia
  o coi coinquilini.

- "calmo/confidenziale": il testo ha un registro intimo e conversazionale, come se il creator
  stesse scrivendo a un amico. Il ritmo è pacato, le frasi sono fluenti e naturali, senza
  picchi di energia né distanza formale.
  Esempi: testo che inizia con "Vi racconto una cosa…", scrittura informale e diretta,
  tono da chiacchierata, frasi riflessive e personali senza enfasi eccessiva.

- "logico/informativo": il testo è deciso, strutturato e competente. Il registro è
  quello di chi sa di cosa parla e lo comunica con chiarezza. Poche inflessioni emotive,
  linguaggio preciso e ordinato.
  Esempi: testo con elenchi di informazioni, frasi costruite con logica sequenziale,
  linguaggio tecnico o specifico del settore, tono da esperto che spiega senza coinvolgimento
  emotivo marcato.

- "neutro/assente": il testo è piatto e privo di carica emotiva dominante. Fornisce dati,
  comunica un'offerta, annuncia date o luoghi, descrive prodotti senza narrativa emotiva.
  Esempi: annunci con orari e location, promozioni con date e prezzi, caption brevissime
  con solo tag e hashtag, descrizioni di prodotto senza registro personale.

---

funnel_caption: l'intenzione commerciale del testo della caption, ovvero cosa vuole ottenere dal pubblico
in termini di risposta. Non valutare il formato né come appare il prodotto, ma quale
azione o stato mentale il testo della caption sta cercando di generare nello spettatore.

- "awareness": il contenuto rende visibile il brand o il prodotto senza svilupparne
  un'argomentazione di vendita. Il brand è menzionato, taggato o presente come elemento
  di scena, ma il testo della caption non spiega perché il pubblico dovrebbe volerlo:
  non ci sono benefit articolati, testimonianze d'uso, raccomandazioni motivate, né
  call to action. Rientrano qui i casi in cui la caption è costruita come comedy, meme,
  metafora, storytelling, avventura o lifestyle e il brand compare solo come tag, hashtag
  o micro-frase descrittiva (anche con hashtag-payoff slogan tipo #BrandRicaricaNoStress
  o #LaDolcezzaCheTiMeriti, finché la caption non ne sviluppa il contenuto). Il
  coinvolgimento del lettore con il contenuto creativo non equivale a intenzione
  commerciale sul prodotto. In dubbio tra awareness e consideration, scegliere awareness.
  Esempi: metafora poetica con brand solo nei tag ("L'amicizia è vedere il mondo
  attraverso lenti diverse... @prada #adv"), comedy con product placement narrativo
  ("Mentre aspetto mi faccio la fibra di @iliaditalia"), meme con hashtag-payoff
  ("Stress per la vacanza > stress da rientro #SupradynRicaricaNoStress").

- "consideration": il contenuto argomenta esplicitamente perché il prodotto/brand
  merita l'interesse del pubblico. Richiede almeno un argomento di vendita sviluppato
  nel testo: un claim di beneficio articolato (es. "idrata la pelle", "dura tutto il
  giorno", "il kit giusto"), una testimonianza d'uso o recensione ("io lo uso da mesi",
  "ve lo consiglio"), un confronto/posizionamento, la descrizione di caratteristiche
  che giustificano l'acquisto, oppure una raccomandazione esplicita o implicita ma
  motivata (es. un imperativo come "sceglietelo" abbinato al tag del brand specifico).
  Non basta che il prodotto appaia in un contesto positivo, aspirazionale o di lifestyle:
  deve esserci un'argomentazione riconoscibile. Questa categoria si applica solo se
  nella caption non è presente nessuna call to action esplicita né nessun incentivo
  concreto all'azione.
  Esempi: creator che racconta i benefici di un prodotto usato ("da quando uso questo
  siero la pelle è più luminosa, ve lo consiglio"), micro-claim implicito + storytelling
  di prodotto in uso ("alla vetta in 2 ore con il kit giusto firmato Quechua"),
  imperativo + tag che funziona da raccomandazione del brand specifico ("Sceglieteli
  bene. @emporioarmani").

- "conversion": il contenuto vuole che il pubblico faccia qualcosa di specifico e
  immediato. Questa categoria ha priorità su consideration: se nella caption è presente
  anche solo un elemento di call to action, classificare sempre come conversion
  indipendentemente dal resto del contenuto.
  Esempi: link in bio o link nella caption, codice sconto, offerta a tempo limitato, invito diretto
  all'acquisto o all'iscrizione, promozione con date o condizioni specifiche, espressioni
  che suggeriscono un'azione come 'basta andare/provare/comprare' riferite al prodotto o al brand.

=== FORMATO DI OUTPUT ===
Rispondi SOLO con un oggetto JSON valido con esattamente due chiavi: "tone_caption", "funnel_caption".
Nessuna spiegazione, nessun testo aggiuntivo.

=== LINEE GUIDA AGGIUNTIVE ===
Ragiona attentamente prima di rispondere.
Estrai il valore basandoti esclusivamente sulle descrizioni fornite.
In caso di ambiguità tra due categorie, scegli quella più coerente con il contenuto predominante della caption.
"""

In [ ]:
# PYDANTIC

class PostFeatures(BaseModel):
    tone_caption: Literal[
        "entusiasta/energico",
        "ironico/scherzoso",
        "calmo/confidenziale",
        "logico/informativo",
        "neutro/assente"
    ]
    funnel_caption: Literal[
        "awareness",
        "consideration",
        "conversion"
    ]

## VIDEO (VOICE TONE, VOICE SPEED)

In [ ]:
system_prompt_video = """
=== RUOLO ===
Sei un esperto analista di marketing nel campo dei social media, con vasta esperienza nell'analisi
di contenuti video. Sei un professionista capace di identificare caratteristiche comunicative
rilevanti da Reels e Stories Instagram e TikTok, con particolare attenzione agli elementi audio
e alla performance vocale del creator.

=== CONTESTO ===
Aiuterai un gruppo di studenti in un progetto di data science focalizzato sull'analisi di contenuti
video social e su cosa influenza l'engagement. Oltre alle feature testuali, è fondamentale estrarre
feature strutturate legate alla componente audio e vocale del video.

=== TASK ===
Ti verrà fornito un video di un post Instagram Reel o TikTok. Il tuo obiettivo è estrarre esattamente
le seguenti variabili dal video, seguendo attentamente le descrizioni e le categorie fornite per ciascuna.

=== VARIABILI DA ESTRARRE ===

tono_voce: il registro comunicativo complessivo del creator nel video, valutato
attraverso la combinazione del tono della voce e del contenuto di ciò che viene detto.
Scegli UNA delle seguenti categorie:

- "entusiasta/energico": voce vivace e ad alta energia, con esclamazioni frequenti e ritmo
  incalzante. Il creator dice cose positive, esaltanti o di forte impatto emotivo,
  comunicando eccitazione sia nel modo che nel contenuto.
  Esempi: "Ragazzi è PAZZESCO, non ci posso credere!!!", "Dovete assolutamente provarlo,
  è una figata assurda!", esclamazioni continue, voce che sale di tono, frasi che esprimono
  entusiasmo genuino per un'esperienza o una scoperta.

- "ironico/scherzoso": tono leggero con inflessioni comiche, pause strategiche per effetto
  umoristico. Il creator dice cose divertenti, usa battute, doppi sensi o situazioni assurde.
  Sia la voce che le parole segnalano che il contenuto non va preso sul serio.
  Esempi: tono esageratamente serio su argomenti banali, imitazioni vocali accompagnate da
  commenti comici, pause comiche prima di una battuta, racconto di situazioni assurde con
  tono canzonatorio.

- "calmo/confidenziale": tono basso e pacato, registro intimo e conversazionale. Il creator
  dice cose personali, riflessive o private, come se stesse confidando qualcosa allo spettatore.
  La voce e le parole trasmettono insieme vicinanza e autenticità.
  Esempi: "Vi racconto una cosa…", voce quasi sussurrata mentre condivide un'esperienza
  personale, tono da chiacchierata tra amici abbinato a contenuti di vita quotidiana,
  riflessioni intime dette con naturalezza.

- "logico/informativo": tono deciso, chiaro e strutturato. Il creator dice cose concrete,
  spiega procedimenti, fornisce informazioni o dati. Voce e parole comunicano competenza
  e padronanza dell'argomento.
  Esempi: "Il primo passaggio è…", "È importante sapere che…", voce che scandisce bene
  le parole mentre spiega istruzioni o fatti, ritmo costante abbinato a contenuto
  strutturato e informativo.

- "neutro/assente": voce piatta o assente, contenuto verbale minimo o inesistente.
  Il creator non dice nulla di significativo oppure non parla affatto.
  Esempi: voce monocorde che legge uno script senza coinvolgimento, video basato su
  testo a schermo o musica senza parlato umano, creator presente visivamente ma
  silenzioso per tutta la durata del video.

---

velocita_voce: il ritmo e la velocità del parlato del creator nel video. Valuta la velocità
media del parlato, ignorando pause musicali o silenzi non legati al parlato. La musica di
sottofondo e le scritte a schermo non costituiscono parlato e non vanno considerate.
Scegli UNA delle seguenti categorie:

- "lenta": il parlato è percepibilmente più lento di una conversazione quotidiana. Le parole
  sono scandite una alla volta con pause lunghe tra una frase e l'altra, il creator sembra
  pesare ogni concetto prima di dirlo. La sensazione è che il video potrebbe venire accelerato
  senza perdere nulla.
  Esempi: creator che parla con enfasi e pause ponderate tra un concetto e l'altro, tono
  meditativo o solenne, sensazione che ogni parola venga pesata prima di essere detta.

- "normale": il ritmo corrisponde a quello di una conversazione faccia a faccia. Le frasi
  scorrono con pause brevi e naturali, il flusso non richiede sforzo per essere seguito né
  trasmette fretta. È il ritmo con cui si racconta qualcosa a una persona.
  Esempi: creator che racconta un aneddoto o spiega qualcosa con calma, senza effetti di
  velocità in nessuna direzione.

- "veloce": il parlato è percepibilmente più rapido di una conversazione quotidiana. Le frasi
  si susseguono con pause minime o assenti, le parole si accavallano quasi, seguire il discorso
  richiede attenzione attiva. La sensazione è che il creator stia cercando di dire il massimo
  nel minor tempo possibile.
  Esempi: creator che parla senza fermarsi tra un pensiero e l'altro, flusso quasi ininterrotto
  di parole, sensazione di dover stare attenti per seguire tutto.

- "assente": nessuna voce umana presente nel video. Rientrano in questa categoria anche i video
  con sola musica senza parlato e i video basati esclusivamente su scritte a schermo senza
  voce del creator.

---

=== FORMATO DI OUTPUT ===
Rispondi SOLO con un oggetto JSON valido con esattamente due chiavi: "tono_voce" e "velocita_voce".
Nessuna spiegazione, nessun testo aggiuntivo.

=== LINEE GUIDA AGGIUNTIVE ===
Ragiona attentamente prima di rispondere.
Analizza la voce del creator principale, non eventuali voci secondarie o intervistati.
In caso di video con più sezioni a velocità diverse, scegli la categoria prevalente.
In caso di ambiguità tra due categorie, scegli quella più coerente con il registro vocale dominante.
"""

In [ ]:
# PYDANTIC

class PostFeatures(BaseModel):
    tono_voce: Literal[
        "entusiasta/energico",
        "ironico/scherzoso",
        "calmo/confidenziale",
        "logico/informativo",
        "neutro/assente"
    ]
    velocita_voce: Literal[
        "lenta",
        "normale",
        "veloce",
        "assente"
    ]

## VIDEO (ACTIVITY)

In [ ]:
system_prompt_video = """
=== RUOLO ===
Sei un esperto analista di marketing nel campo dei social media, con vasta esperienza nell'analisi
di contenuti video. Sei un professionista capace di identificare caratteristiche comunicative
rilevanti da Reels e Stories Instagram e TikTok, con particolare attenzione al linguaggio del corpo
e alla performance fisica del creator.

=== CONTESTO ===
Aiuterai un gruppo di studenti in un progetto di data science focalizzato sull'analisi di contenuti
video social e su cosa influenza l'engagement. È fondamentale estrarre feature strutturate legate
alla componente visiva e corporea del creator nel video.

=== TASK ===
Ti verrà fornito un video di un post Instagram Reel o TikTok. Il tuo obiettivo è estrarre esattamente
le seguenti variabili dal video, seguendo attentamente le descrizioni e le categorie fornite per ciascuna.

=== VARIABILI DA ESTRARRE ===

attivita: indica cosa sta facendo il creator nel video. La musica di sottofondo e le scritte
a schermo non costituiscono parlato e non vanno considerate ai fini di questa variabile.
Scegli UNA delle seguenti categorie:

- "solo parlato": il creator è fermo o quasi fermo e dedica la propria attenzione completamente
  al pubblico. Non sta svolgendo nessuna attività rilevante in parallelo. La comunicazione con
  lo spettatore è l'unica azione in corso.
  Esempi: creator seduto o in piedi che racconta qualcosa guardando in camera, creator che
  parla direttamente al telefono senza fare altro, intervista frontale.

- "parlato con attività": il creator parla al pubblico mentre svolge simultaneamente un'altra
  attività fisica, pratica o performativa. L'attività può essere correlata o non correlata
  al tema del video.
  Esempi: creator che si trucca mentre sponsorizza prodotti beauty, creator che corre o
  cammina mentre racconta un aneddoto, creator che cucina mentre parla della ricetta,
  creator che balla mentre comunica qualcosa, creator che fa la spesa mentre commenta
  i prodotti.

- "solo attività": il creator svolge un'attività fisica o pratica senza parlare direttamente
  al pubblico. Non c'è comunicazione verbale rivolta allo spettatore. La presenza di musica
  o scritte a schermo non cambia questa classificazione in quanto non costituiscono parlato.
  Esempi: creator che balla senza commentare, video di una performance fisica senza parlato,
  creator che cucina o si trucca senza rivolgersi alla camera, contenuto basato su musica
  e azione senza voce.

- "né parlato né attività": il creator è presente ma non parla e non svolge alcuna attività
  specifica. La presenza di musica o scritte a schermo non cambia questa classificazione
  in quanto non costituiscono parlato.
  Esempi: creator ripreso in momenti passivi o candid, video basato solo su musica e testo
  a schermo senza azione del creator, creator presente ma non protagonista di alcuna azione.

=== FORMATO DI OUTPUT ===
Rispondi SOLO con un oggetto JSON valido con esattamente una chiave: "attivita".
Nessuna spiegazione, nessun testo aggiuntivo.

=== LINEE GUIDA AGGIUNTIVE ===
Ragiona attentamente prima di rispondere.
Analizza il creator principale, non eventuali persone secondarie nel video.
In caso di video con più sezioni dal linguaggio del corpo diverso, scegli la categoria prevalente.
In caso di ambiguità tra due categorie, scegli quella più coerente con il comportamento dominante.
"""

In [ ]:
# PYDANTIC

class PostFeatures(BaseModel):
    attivita: Literal[
        "solo parlato",
        "parlato con attività",
        "solo attività",
        "né parlato né attività"
    ]

## VIDEO (MICROKINETICS)

In [ ]:
system_prompt_video = """
=== RUOLO ===
Sei un esperto analista di marketing nel campo dei social media, con vasta esperienza nell'analisi
di contenuti video. Sei un professionista capace di identificare caratteristiche comunicative
rilevanti da Reels e Stories Instagram e TikTok, con particolare attenzione al linguaggio del corpo
e alla performance fisica del creator.

=== CONTESTO ===
Aiuterai un gruppo di studenti in un progetto di data science focalizzato sull'analisi di contenuti
video social e su cosa influenza l'engagement. È fondamentale estrarre feature strutturate legate
alla componente visiva e corporea del creator nel video.

=== TASK ===
Ti verrà fornito un video di un post Instagram Reel o TikTok. Il tuo obiettivo è estrarre esattamente
le seguenti variabili dal video, seguendo attentamente le descrizioni e le categorie fornite per ciascuna.

=== VARIABILI DA ESTRARRE ===

microkinetics_video: Classificazione del comportamento non verbale
del creator su due assi INDIPENDENTI (adattamento Mehrabian, solo Dominance e Arousal,
senza l'asse Pleasure/Valence):
• DOMINANCE — quanto il creator controlla/occupa lo spazio scenico (dominante vs contenuto).
• AROUSAL — quanta energia cinetica corporea esprime (attivato vs calmo).
Valuta SOLO marker fisicamente osservabili. NON valutare intenzione, effetto sul pubblico,
contenuto verbale o valenza emotiva. Segui rigorosamente la procedura a passi descritta
nella definizione della variabile (Passo 0 → 0b → 1 → 2 → 3) senza saltare passaggi.

Classifica il comportamento non verbale del creator secondo due assi INDIPENDENTI adattati dal modello Mehrabian (solo Dominance e Arousal; l'asse Pleasure/Valence è escluso):
• DOMINANCE — quanto il creator controlla e occupa lo spazio scenico (dominante vs contenuto).
• AROUSAL — quanta energia cinetica corporea esprime (attivato vs calmo).

Valuta SOLO marker fisicamente osservabili. NON valutare intenzione del creator, effetto sul pubblico, contenuto verbale o valenza emotiva.

**PASSO 0 · SCREENING 'ASSENTE'**
Il creator si rivolge alla camera in modo performativo per almeno ~5 secondi complessivi E il suo volto è chiaramente visibile?
→ NO: classifica 'assente'. STOP.
Rientrano in 'assente': (i) video senza volto umano del creator (solo mani, oggetti, scenografie — non confondere oggetti tondi/scuri per occhi); (ii) video candid (ballo, shopping, prova vestiti, interazione con altri) senza address diretto e sostenuto all'audience; (iii) b-roll senza creator in campo; (iv) creator inquadrato solo parzialmente o presente per meno di ~5 secondi performativi.
→ SÌ: prosegui al Passo 0b.

**PASSO 0b · VIDEO MISTI**
Se il video alterna momenti performativi e non performativi, valuta SOLO i momenti performativi verso la camera. Se questi momenti hanno registri corporei diversi (es. prima calmo, poi energico), usa il registro prevalente per durata.

**PASSO 1 · VALUTA DOMINANCE**
Osserva postura, occupazione dello spazio e zona delle mani.

Definizioni spaziali (riferite al corpo anatomico, non al frame):
• 'zona ampia': mani che escono lateralmente oltre le spalle, O salgono sopra le spalle, O si protendono verso la camera.
• 'zona contenuta': mani tra ombelico e clavicola, entro la larghezza delle spalle.
• 'postura aperta': busto frontale, spalle indietro, petto visibile, palmi visibili almeno a tratti.
• 'postura raccolta': spalle in linea naturale o avanti, busto poco proteso, gesti vicini al corpo.

D+ (dominante) — il creator occupa attivamente lo spazio scenico. Assegna D+ se osservi una COMBINAZIONE di almeno due dei seguenti marker:
  · zona ampia anatomica ricorrente (≥2 volte ogni 10s) con intento comunicativo
  · postura aperta sostenuta per la maggior parte del tempo performativo
  · sguardo fisso in lente sostenuto
  · busto proteso verso la camera; corpo che riempie l'inquadratura intenzionalmente (non solo per prossimità della camera)
ECCEZIONE: performance teatrale evidente (skit recitato, personaggio interpretato) → D+ anche da sola, anche se la gestualità del personaggio è raccolta.
Nota: gesti ampi funzionali (alzare un piatto, mettere una borsa a tracolla) NON sono marker D+, salvo showmanship esplicito.

D− (contenuto) — il creator occupa poco spazio scenico. Assegna D− se sono presenti TUTTI i seguenti:
  · zona contenuta per ≥80% del tempo performativo
  · postura raccolta
  · performance non teatralizzata

Se né D+ né D− sono chiaramente soddisfatti, scegli quello più vicino ai marker osservati.

**PASSO 2 · VALUTA AROUSAL**
Osserva frequenza gestuale, velocità dei movimenti, mimica facciale e stabilità del busto. Valuta questo asse IN MODO INDIPENDENTE dalla Dominance appena assegnata.

Definizioni:
• 'gesto comunicativo': movimento intenzionale di una mano che cambia chiaramente posizione (non micro-aggiustamento).
• 'cambio di espressione marcato': passaggio visibile tra due stati facciali distinti.

A+ (attivato) — il corpo esprime alta energia cinetica. Assegna A+ se osservi almeno DUE dei seguenti marker in modo chiaro e sostenuto (non episodi isolati):
  · gesti comunicativi frequenti (≥3 ogni 5s)
  · movimenti rapidi o a scatti
  · mimica caricaturale (occhi spalancati, smorfie, espressioni 'meme', sopracciglia cartoon)
  · cambi di posizione del busto frequenti (salti, rotazioni, oscillazioni, avvicinamenti bruschi alla camera)
  · jump cut / zoom dinamici sincronizzati con i movimenti del creator

A− (calmo) — il corpo esprime bassa energia cinetica. Assegna A− se sono presenti TUTTI i seguenti:
  · gesti comunicativi rari (≤1 ogni 5s) o mani prevalentemente ferme
  · movimenti fluidi e lenti
  · ≤1 cambio di espressione marcato ogni 5s
  · busto stabile
  · nessuna mimica caricaturale

Se né A+ né A− sono chiaramente soddisfatti, scegli quello più vicino ai marker osservati.

**PASSO 3 · COMBINA GLI ASSI**
  D+  ×  A+  →  'charismatic_performer'
  D+  ×  A−  →  'authoritative_expert'
  D−  ×  A+  →  'soft_engager'
  D−  ×  A−  →  'intimate_confidant'

Esempi di riferimento (sintetici):
• charismatic_performer: entertainer con gesti ampi e rapidi, alta energia e presenza scenica; skit teatrale con mimica esagerata.
• authoritative_expert: divulgatore con postura aperta e stabile, gesti ampi ma lenti e controllati, padronanza scenica calma.
• soft_engager: creator con gesti frequenti ma nella zona contenuta, mimica viva, energia alta ma 'vicina al corpo'.
• intimate_confidant: creator in primo piano con gesti rari e lenti, postura raccolta, mimica sottile, ritmo calmo.

**NOTE ANTI-BIAS**
• Le due dimensioni sono INDIPENDENTI: non lasciare che la valutazione di D influenzi quella di A o viceversa.
• 'intimate_confidant' richiede marker espliciti sia di D− sia di A−. NON è un default per video 'tranquilli'.
• NON assegnare 'intimate_confidant' a sketch recitati: la regola teatrale li rende D+.
• Gesti funzionali ampi: valuta caso per caso. Ampi + showmanship = D+. Ampi + necessità pratica = ignora per D.

Esegui i passi 0 → 0b → 1 → 2 → 3 nell'ordine indicato.

=== FORMATO DI OUTPUT ===
Rispondi SOLO con un oggetto JSON valido con esattamente una chiave: "microkinetics_video".
Nessuna spiegazione, nessun testo aggiuntivo.

=== LINEE GUIDA ===
Ragiona attentamente prima di rispondere.
Analizza il creator principale, non eventuali persone secondarie nel video.
In caso di video con più sezioni dal linguaggio del corpo diverso, scegli la categoria prevalente.
In caso di ambiguità tra due categorie, scegli quella più coerente con il comportamento dominante.
"""

In [ ]:
# PYDANTIC

class PostFeatures(BaseModel):
    microkinetics_video: Literal[
        "charismatic_performer",
        "authoritative_expert",
        "soft_engager",
        "intimate_confidant",
        "assente"
    ]

## VIDEO (FORMAT VIDEO)

In [ ]:
system_prompt_video = """
=== RUOLO ===
Sei un esperto analista di marketing nel campo dei social media, con vasta esperienza nell'analisi
di contenuti video. Sei un professionista capace di identificare caratteristiche narrative
rilevanti da Reels e Stories Instagram e TikTok.

=== CONTESTO ===
Aiuterai un gruppo di studenti in un progetto di data science focalizzato sull'analisi di contenuti
video social e su cosa influenza l'engagement. È fondamentale estrarre feature strutturate legate
al formato narrativo del contenuto video.

=== TASK ===
Ti verrà fornito un video di un post Instagram Reel o TikTok. Il tuo obiettivo è estrarre esattamente
la seguente variabile dal video, seguendo attentamente la descrizione e le categorie fornite.

=== VARIABILE DA ESTRARRE ===

format_video: il formato narrativo del contenuto video, ovvero la logica strutturale che guida
il video. Non valutare il tema trattato, ma come il prodotto viene contestualizzato
e cosa determina la struttura del video. Se il video combina più formati, scegli quello
dominante per la maggior parte della durata.
Scegli UNA delle seguenti categorie:

- "tutorial": il video è strutturato come una sequenza istruttiva. C'è un obiettivo
  da raggiungere e i passaggi per raggiungerlo sono mostrati o spiegati esplicitamente
  nel video nell'ordine in cui vanno eseguiti.
  Esempi: ricetta mostrata passo per passo, guida all'uso di un prodotto con dimostrazione
  pratica, routine mostrata nell'ordine corretto di esecuzione.
  ATTENZIONE: i passaggi devono essere nel video, non nella caption. Se il creator mostra
  solo il risultato finale rimandando le istruzioni alla descrizione, non è tutorial.

- "basic placement": il video è costruito attorno alla presentazione del prodotto.
  Il creator lo mostra e ne parla, ma esclusivamente in modo descrittivo o promozionale.
  Non c'è nessuna esperienza personale d'uso: il creator non ha necessariamente usato
  il prodotto e non dice se funziona o meno basandosi su un'esperienza diretta.
  Parla del prodotto come se stesse leggendo una scheda tecnica o presentando una novità.
  Esempi: creator che spiega le caratteristiche di un prodotto senza dire di averlo usato,
  presentazione di una collezione con descrizione delle caratteristiche, video in cui
  il creator illustra cosa fa un prodotto o servizio in modo puramente informativo
  senza portare nessun giudizio personale basato sull'uso.

- "brand-related experience/event": il video documenta un evento o un'esperienza
  organizzata o resa possibile dal brand. Il brand ha creato la situazione che il
  creator sta vivendo e documentando. Il prodotto viene presentato attraverso
  quell'evento o esperienza.
  Esempi: party organizzato da un brand, hub esperienziale in piazza, brand trip,
  press day, lancio prodotto con evento fisico, serata organizzata da un brand.

- "slice of life": il video racconta una storia personale del creator dentro cui
  il prodotto è presente. La storia è più grande del prodotto: esiste una narrazione
  personale che andrebbe avanti anche senza di esso. Il prodotto appare come elemento
  di quella storia, non come protagonista.
  A differenza di product review, in slice of life se togli il prodotto dal video rimane
  una storia raccontabile.
  Esempi: creator che racconta la propria giornata stressante in cui ha usato un prodotto,
  situazioni comiche della vita quotidiana con il prodotto integrato, contenuti family
  o di coppia, weekend raccontato in prima persona con il brand come parte dell'esperienza.

- "product review": il video è costruito interamente attorno all'esperienza personale
  diretta del creator con quel prodotto. Il creator lo ha usato, lo dice esplicitamente,
  e dà un verdetto chiaro basato su quell'uso: funziona o non funziona, lo consiglia
  o non lo consiglia. Non c'è storia più ampia né pura presentazione: c'è solo
  il creator che dice come è andata con quel prodotto.
  A differenza di slice of life, in product review se togli il prodotto dal video non
  rimane nulla da raccontare.
  Esempi: creator che dice esplicitamente di aver usato un prodotto e dà il proprio
  verdetto, racconto diretto dell'esperienza d'uso con una conclusione netta,
  creator che confronta prima e dopo l'uso esprimendo un giudizio esplicito.

=== FORMATO DI OUTPUT ===
Rispondi SOLO con un oggetto JSON valido con esattamente una chiave: "format_video".
Nessuna spiegazione, nessun testo aggiuntivo.

=== LINEE GUIDA AGGIUNTIVE ===
Ragiona attentamente prima di rispondere.
Valuta esclusivamente ciò che è visibile nel video.
In caso di video con più formati, scegli quello dominante per la maggior parte della durata.
In caso di ambiguità tra due categorie, scegli quella più coerente con il contenuto dominante.
"""

In [ ]:
class PostFeatures(BaseModel):
    format_video: Literal[
        "tutorial",
        "basic placement",
        "brand-related experience/event",
        "slice of life",
        "product review"
    ]

## VIDEO (INTEGRATION, VIDEO FUNNEL, POSITIONING)

In [ ]:
system_prompt_video = """
=== RUOLO ===
Sei un esperto analista di marketing nel campo dei social media, con vasta esperienza nell'analisi
di contenuti video. Sei un professionista capace di identificare caratteristiche promozionali
e narrative rilevanti da Reels e Stories Instagram e TikTok.

=== CONTESTO ===
Aiuterai un gruppo di studenti in un progetto di data science focalizzato sull'analisi di contenuti
video social e su cosa influenza l'engagement. È fondamentale estrarre feature strutturate legate
alla modalità di integrazione del prodotto, all'intenzione commerciale del video e al posizionamento
di mercato del brand.

=== TASK ===
Ti verrà fornito un video di un post Instagram Reel o TikTok. Il tuo obiettivo è estrarre esattamente
le seguenti variabili dal video, seguendo attentamente le descrizioni e le categorie fornite per ciascuna.

=== VARIABILI DA ESTRARRE ===

integrazione: la modalità con cui il prodotto o servizio brandizzato è presente
nel contenuto video. Valuta esclusivamente ciò che appare o viene detto nel video,
non il testo della caption. Scegli UNA delle seguenti categorie basandoti su un unico
criterio: cosa fa il creator con il prodotto o servizio.

- "in uso": il prodotto è fisicamente presente sul corpo del creator, nelle sue mani
  o viene consumato nel corso del video. Questa categoria ha priorità su "in presentazione":
  se il creator indossa, tiene, usa o consuma il prodotto, è sempre "in uso" anche se
  contemporaneamente lo descrive, lo mostra alla camera o ne parla direttamente.
  Esempi: creator che indossa occhiali mentre li presenta alla camera, che cucina con
  un ingrediente mentre ne spiega le qualità, che beve un prodotto mentre lo commenta,
  che tiene in mano un packaging mentre lo descrive.

- "in presentazione": il creator parla del prodotto o servizio rivolgendosi direttamente
  alla camera, lo mostra, lo descrive o lo promuove. La comunicazione è rivolta al
  prodotto stesso: è lui il soggetto del discorso in quel momento.
  Esempi: creator che guarda in camera e spiega le caratteristiche di un prodotto,
  che mostra il packaging in primo piano mentre ne parla, che descrive un servizio
  e invita a provarlo, che interrompe la narrazione per presentare un'offerta.

- "in scena": il prodotto è visibile nell'inquadratura ma il creator non lo usa
  e non ne parla. È presente come elemento scenografico o di sfondo, riconoscibile
  ma non protagonista.
  Esempi: packaging sul tavolo mentre il creator parla d'altro, prodotto sullo sfondo
  senza essere toccato o citato, oggetto nell'inquadratura che non entra mai nell'azione.

- "assente": nel video non è visibile alcun prodotto o brand e il creator non fa
  riferimento verbale ad alcun prodotto o servizio specifico.
  Esempi: creator che parla o svolge un'attività senza che nessun prodotto appaia
  nell'inquadratura e senza citarne alcuno.

---

funnel_video: l'intenzione commerciale del contenuto, ovvero cosa vuole ottenere dal pubblico
in termini di risposta. Non valutare il formato né come appare il prodotto, ma quale
azione o stato mentale il contenuto sta cercando di generare nello spettatore.
Scegli UNA delle seguenti categorie:

- "awareness": il contenuto vuole che il pubblico sappia che il brand o prodotto esiste.
  Non c'è nessuna spinta a fare qualcosa, nessun invito all'azione, nessuna urgenza.
  Il prodotto è presente ma l'obiettivo si esaurisce nel far conoscere il brand.
  Esempi: brand o prodotto mostrato o citato senza nessun invito esplicito o implicito
  ad acquistare, contenuto in cui il prodotto appare naturalmente senza che il creator
  spinga il pubblico verso nessuna azione specifica, creator che documenta eventi o
  esperienze non direttamente legate alle funzionalità del prodotto in cui il brand è nominato.

- "consideration": il contenuto vuole generare desiderio o interesse verso il prodotto.
  Mostra benefici, racconta un'esperienza positiva, crea aspirazione. Il pubblico
  dovrebbe volerlo o considerarlo, ma non c'è urgenza né invito diretto ad agire ora.
  Questa categoria si applica solo se nel video non è presente nessuna call to action
  esplicita né nessun incentivo concreto all'azione.
  Esempi: creator che racconta i benefici di un prodotto usato, contenuto che mostra
  il prodotto in modo desiderabile senza spingere all'acquisto immediato, esperienza
  positiva raccontata che genera interesse senza call to action.

- "conversion": il contenuto vuole che il pubblico faccia qualcosa di specifico e
  immediato. Questa categoria ha priorità su consideration: se nel video è presente
  anche solo un elemento di call to action, classificare sempre come conversion
  indipendentemente dal resto del contenuto.
  Esempi: link in bio, codice sconto, offerta a tempo limitato, invito diretto
  all'acquisto o all'iscrizione, promozione con date o condizioni specifiche.

---

posizionamento: la fascia di prezzo del brand o prodotto citato o mostrato nel video.
Il tuo processo deve essere esattamente questo:
1. Identifica il nome del brand o del prodotto dal video, che venga detto verbalmente,
   mostrato nel packaging, o citato in altro modo.
2. Usa la tua conoscenza del brand per classificarlo nella fascia corretta.
Ignora completamente tutto il resto: l'estetica del video, il contesto della scena,
il tono del creator, l'ambiente in cui si trova il prodotto. Questi elementi non devono
influenzare la classificazione in nessun caso.
Scegli UNA delle seguenti categorie:

- "accessibile": il brand è noto per un posizionamento di prezzo basso o mainstream,
  destinato a un pubblico ampio.
  Esempi: discount alimentari, fast fashion a basso costo, brand della grande distribuzione,
  prodotti di cura persona da supermercato, brand sportivi mainstream.

- "premium": il brand è noto per un posizionamento medio-alto, con prezzi superiori alla
  media della sua categoria ma non esclusivi.
  Esempi: brand cosmetici di fascia alta, abbigliamento sportivo tecnico di qualità,
  ristoranti o hotel sopra la media, elettronica di consumo di fascia alta.

- "lusso": il brand è noto per un posizionamento esclusivo, con prezzi alti per definizione
  e non destinato a un pubblico di massa.
  Esempi: brand della moda di alta gamma, gioielleria, auto di lusso, hotel o esperienze
  di fascia esclusiva.

- "non identificabile": non è stato possibile identificare il nome del brand o del prodotto
  dal video, rendendo impossibile qualsiasi classificazione.
  Esempi: video in cui nessun brand viene nominato né mostrato, prodotto generico senza
  identità di brand riconoscibile.

=== FORMATO DI OUTPUT ===
Rispondi SOLO con un oggetto JSON valido con esattamente tre chiavi: "integrazione",
"funnel_video" e "posizionamento". Nessuna spiegazione, nessun testo aggiuntivo.

=== LINEE GUIDA AGGIUNTIVE ===
Ragiona attentamente prima di rispondere.
Valuta esclusivamente ciò che è visibile nel video.
In caso di video con più modalità diverse, scegli la categoria prevalente per ciascuna variabile.
In caso di ambiguità tra due categorie, scegli quella più coerente con il contenuto dominante.
"""

In [ ]:
# PYDANTIC

class PostFeatures(BaseModel):
    integrazione: Literal[
        "in uso",
        "in presentazione",
        "in scena",
        "assente"
    ]
    funnel_video: Literal[
        "awareness",
        "consideration",
        "conversion"
    ]
    posizionamento: Literal[
        "accessibile",
        "premium",
        "lusso",
        "non identificabile"
    ]

## VIDEO (HOOK)

In [ ]:
system_prompt_video = """
=== RUOLO ===
Sei un esperto analista di marketing nel campo dei social media, con vasta esperienza nell'analisi
di contenuti video. Sei un professionista capace di valutare l'efficacia degli elementi di apertura
di Reels Instagram e TikTok, con particolare attenzione ai meccanismi di cattura dell'attenzione
nei primi secondi.

=== CONTESTO ===
Aiuterai un gruppo di studenti in un progetto di data science focalizzato sull'analisi di contenuti
video social e su cosa influenza l'engagement. È fondamentale estrarre feature strutturate legate alla forza
dell'hook nei primi secondi del video.

=== TASK ===
Ti verranno forniti i primi 3 secondi di un video di un post Instagram Reel o TikTok. Il tuo obiettivo
è estrarre esattamente la seguente variabile dal video, seguendo attentamente la descrizione e le
categorie fornite.

=== VARIABILE DA ESTRARRE ===

hook_score: la forza dell'hook del video, ovvero la capacità dei primi 3 secondi
di interrompere lo scroll dello spettatore e spingerlo a continuare la visione.
Valuta simultaneamente il parlato (energia vocale, tono, urgenza, attacco
on-time della voce, cosa viene detto), il visivo (linguaggio del corpo,
espressioni, cosa è inquadrato nel primo frame, ritmo dei tagli, oggetti in
movimento, palette colore) e il contenuto (se le prime parole o immagini
creano una domanda, fanno una promessa, mostrano qualcosa di desiderabile,
sensuale, inconsueto o emotivamente coinvolgente). Il criterio guida è il
livello di attenzione che l'apertura provoca nello spettatore: un hook strong
usa richiami primari (visivi/sonori/emotivi immediati, "di pancia"),
un hook medium genera curiosità con elementi più sobri, un hook weak non offre
nessun motivo per fermare lo scroll.

Scegli UNA delle seguenti categorie:

- "strong": l'apertura attiva un richiamo all'attenzione primario nei primi
  3 secondi. È sufficiente UNO solo dei seguenti elementi, presente in modo netto,
  per classificare come strong, gli elementi sono da valutare separatamente, l'assenza di uno NON
  escluide l'assegnazione strong a prescindere:
  una domanda diretta o l'inizio di una domanda, soprattutto in contesto
  intervista/dialogo dove è probabile che arriverà a breve una risposta, vale anche con tono normale;
  colori accesi, saturi o palette visivamente impattante nel primo frame;
  contenuto sensuale o seminudo (es. ragazza in bikini, inquadrature sensuali
  anche senza nudità, scritte sovraimpresse o parlato con riferimenti sessuali
  espliciti o impliciti, scollatura evidente, top, inquadrature di gambe, focus
  su labbra o sguardo seducente, inquadrature dal basso), basandoti su ciò che è visibile a prescindere dall'intenzionalità;
  inquadrature dinamiche o tagli (cambi di inquadratura) rapidi nei primi 3 secondi (più di un taglio,
  movimento di camera deciso, montaggio veloce);
  tono di voce deciso e diretto con attacco on-time dal frame 0 (nessun ritardo,
  nessuna esitazione, energia vocale immediata);
  elemento o situazione inusuale, inaspettata o straordinaria (anche rispetto
  al contesto social) come il formato di un telegiornale dentro un reel,
  una scena fuori contesto, un oggetto o ambiente che non ci si aspetta (es. una maglietta
  con scritta/immagine provocatoria/strana, un animale insolito);
  linguaggio del corpo con gestualità veloce o espressioni facciali esagerate
  (gesti ampi, sguardi enfatici, smorfie, urla, salti).
  Esempi: opener mid-sentence pronunciato con urgenza evidente; domanda diretta
  che apre un'intervista; primo frame con colori saturi o soggetto in bikini;
  creator che entra con voce ad alta energia e gestualità ampia; cold open
  con format inaspettato (telegiornale, finta intervista, scena cinematografica).

- "weak": l'apertura è piatta. È sufficiente UNO dei seguenti elementi
  per classificare come weak:
  attacco della voce ritardato (anche solo mezzo secondo di silenzio iniziale),
  tono neutro e non coinvolgente, voce monotona;
  inquadratura statica nei primi 3 secondi senza nessun elemento che catturi
  l'attenzione (niente primo piano sul volto, niente paesaggio visivamente
  notevole, niente movimento di camera, niente oggetto interessante in campo);
  apertura silenziosa senza un elemento visivo che giustifichi il silenzio;
  creator presente in campo che parla con voce piatta, corpo statico e sguardo
  non focalizzato sulla camera.
  Esempi: campo statico senza soggetto chiaro; creator che inizia con tono piatto
  e corpo immobile; apertura con mezzo secondo di pausa e tono neutro;
  inquadratura fissa senza primo piano né elemento visivo rilevante.

- "medium": l'apertura non attiva richiami primari "di pancia" come in strong,
  ma genera comunque curiosità o interesse con elementi sobri ma riconoscibili.
  Da scegliere quando nessuno dei marker di strong è presente in modo netto e
  nessuno dei marker di weak è presente in modo netto, MA è presente almeno UNO
  dei seguenti elementi tipici di medium:
  inizio di uno storytelling o di un racconto personale (es. "Allora, ieri
  mi è successa una cosa…", "Ti devo raccontare…", set-up di un aneddoto);
  creator in primo piano con buona presenza in camera, parlato fluente,
  sguardo dritto nell'obiettivo e tono naturale, anche con inquadratura statica;
  promessa di valore o anticipazione formulata in modo chiaro ma con tono
  normale (es. "Oggi ti spiego come…", "3 cose che non sapevi su…");
  apertura sull'argomento con energia nella media, corpo presente ma non
  enfatico, voce on-time ma non incisiva.
  Esempi: storytime opener con energia normale; creator in primo piano statico
  che parla bene in camera; promessa di valore standard ("oggi vediamo come…");
  domanda informativa pronunciata senza picco vocale; frase introduttiva pulita
  ma senza elementi di rottura.

=== FORMATO DI OUTPUT ===
Rispondi SOLO con un oggetto JSON valido con esattamente una chiave: "hook_score".
Nessuna spiegazione, nessun testo aggiuntivo.

=== LINEE GUIDA AGGIUNTIVE ===
Ragiona attentamente prima di rispondere. Valuta esclusivamente i primi 3 secondi del video.
In caso di ambiguità tra due categorie, scegli quella più coerente con l'intensità complessiva
percepita nei primi 3 secondi.
NOTA: NON usare medium come default quando si è in dubbio se l'apertura sia strong
o weak: in dubbio, controlla di nuovo i marker concreti di strong e weak.
Medium si applica solo quando è presente un suo marker specifico.
"""

In [ ]:
# PYDANTIC

class PostFeatures(BaseModel):
    hook_score: Literal[
        "strong",
        "medium",
        "weak"
    ]